# Pré processamento

**Projeto 1 – Explainable AI (2025.2)**

**Aluno:** Maurício Sightman G. A. Leandro

### *Overview*

O objetivo do Projeto 1 é estudar a interpretabilidade de modelos black-box de *machine learning*. Para isso, foi escolhido um *dataset* (descrito abaixo) e treinado um modelo utilizando a técnica Random Forest. Foram aplicadas as seguintes técnicas de interpretabilidade: Global Surrogate, Permutation Feature Importance, Partial Dependence Plots, LIME e SHAP.

O *dataset* escolhido foi o [TOW-IDS](https://ocslab.hksecurity.net/Datasets/tow-ids-automotive-ethernet-intrusion-dataset), que compreende dados referentes a cinco tipos de ataques em fluxos de rede Ethernet automotiva. Os dados estão em formato *.pcap* e envolvem diferentes protocolos de rede, como AVTP, PTP e CAN sobre UDP. Dessa forma, é necessário realizar um pré-processamento para a extração de *features*.

Nesse contexto, foi utilizado um algoritmo simplificado proposto em [*Multi-stage deep learning-based intrusion detection system for automotive Ethernet networks*](https://www.sciencedirect.com/science/article/abs/pii/S1570870524001598). Nesse algoritmo, os dados são organizados em janelas temporais, de modo que a detecção de certos tipos de ataques (como *CAN Replay* e *Frame Injection*) depende da informação temporal entre os pacotes.

No final do processo, é gerado um vetor de tamanho 58, que corresponde à soma das diferenças de cada pacote dentro da janela. Intuitivamente, essas *features* medem a variabilidade dos primeiros bytes que compõem os cabeçalhos dos protocolos. Essas informações refletem as características de controle dos pacotes em relação à rede.


## Setup

In [ ]:
import torch
import time
import random
from typing import Tuple
from tqdm import tqdm

import numpy as np
import pandas as pd
from scapy.all import raw, PcapReader, Ether
import seaborn as sns
import matplotlib.pyplot as plt

import time
from typing import Counter, Tuple
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import confusion_matrix

In [ ]:
DEFAULT_SEED = 10

torch.manual_seed(DEFAULT_SEED)
torch.cuda.manual_seed_all(DEFAULT_SEED)
torch.cuda.manual_seed(DEFAULT_SEED)
torch.mps.manual_seed(DEFAULT_SEED)
np.random.seed(DEFAULT_SEED)
random.seed(DEFAULT_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
train_x_path    = "data/tow-ids-dataset/raw/Automotive_Ethernet_with_Attack_original_10_17_19_50_training.pcap"
train_y_path    = "data/tow-ids-dataset/raw/y_train.csv"
number_of_bytes = 58
window_size     = 64
window_stride   = 1

In [ ]:
def get_overall_metrics(y_true, y_pred):
  tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
  acc = (tp+tn)/(tp+tn+fp+fn)
  tpr = tp/(tp+fn)
  fpr = fp/(fp+tn)
  precision = tp/(tp+fp)
  f1 = (2*tpr*precision)/(tpr+precision)
  return {'acc':acc,'tpr':tpr,'fpr':fpr,'precision':precision,'f1-score':f1}

def plot_confusion_matrix(y_true, y_pred):
  cm = confusion_matrix(y_true, y_pred)
  group_counts = [f'{value:.0f}' for value in confusion_matrix(y_true, y_pred).ravel()]
  group_percentages = [f'{value*100:.2f}%' for value in confusion_matrix(y_true, y_pred).ravel()/np.sum(cm)]
  labels = [f'{v1}\n{v2}' for v1, v2 in zip(group_counts, group_percentages)]
  labels = np.array(labels).reshape(2,2)
  sns.heatmap(cm, annot=labels, cmap='Oranges', xticklabels=['Predicted Benign', 'Predicted Malicious'], yticklabels=['Actual Benign', 'Actual Malicious'], fmt='')
  return

def get_tpr_per_attack(y_labels, y_pred):
  aux_df = pd.DataFrame({'Label':y_labels,'prediction':y_pred})
  total_per_label = aux_df['Label'].value_counts().to_dict()
  correct_predictions_per_label = aux_df.query('Label != "Normal" and prediction == True').groupby('Label').size().to_dict()
  tpr_per_attack = {}
  for attack_label, total in total_per_label.items():
    if attack_label == 'Normal':
      continue
    tp = correct_predictions_per_label[attack_label] if attack_label in correct_predictions_per_label else 0
    tpr = tp/total
    tpr_per_attack[attack_label] = tpr
  return tpr_per_attack

## Carregando Conjunto de Dados

In [ ]:
def detect_protocol_scapy(pkt):
    """Detect protocol using Scapy's layer inspection."""
    eth_type = pkt[Ether].type

    if eth_type in [2054, 35061, 8938]:
        return 'L2'

    if eth_type in [2048, 34525]:
        return 'IP_UDP'
        
    if eth_type in [33024, 8944]:
        return 'AVTP'
        
    if eth_type in [35063]:
        return 'PTP'
        
    return '-1'

def load(raw_x_path: str, raw_y_path: str, number_of_bytes: int = 58, protocol_filter: str = None) -> Tuple[np.ndarray, pd.DataFrame]:
    """Load TOW-IDS dataset"""
    print("Loading raw data...")
    start_time = time.time()

    labels = pd.read_csv(raw_y_path, header=None, names=["pkt_idx", "class", "label"])
    labels['label'] = labels['label'].map({
        'Normal': 'Normal',
        'C_D': 'CAN DoS',
        'P_I': 'PTP Sync',
        'M_F': 'Switch MAC Flooding',
        'F_I': 'Frame Injection',
        'C_R': 'CAN Replay',
    })

    n = len(labels)
    values      = np.empty((n, number_of_bytes), dtype=np.float32)
    timestamps  = np.empty(n, dtype=object)
    protocols   = np.empty(n, dtype=object)

    with PcapReader(raw_x_path) as pcap_reader:
        for i, pkt in tqdm(enumerate(pcap_reader), total=n):
            protocol = detect_protocol_scapy(pkt)
            if (protocol_filter and (
                (type(protocol_filter) == list and protocol not in protocol_filter) or 
                (type(protocol_filter) != list and protocol != protocol_filter)
                )):
                continue

            b = raw(pkt)
            m = min(len(b), number_of_bytes)
            arr = np.frombuffer(b, dtype=np.uint8, count=m)
            if len(b) < number_of_bytes:
                arr = np.pad(arr, (0, number_of_bytes - len(b)), 'constant')
            
            protocols[i]    = protocol
            timestamps[i]   = pkt.time
            values[i]       = arr

    values /= 255.0

    labels['timestamp'] = timestamps
    labels['protocol'] = protocols
    labels.dropna(inplace=True)

    valid_idx = labels.index
    values = values[valid_idx]

    print(f"Loading raw data finished in {(time.time() - start_time):.2f}s with shape of {values.shape}")
    return values, labels

In [ ]:
raw_X_train, raw_y_train = load(train_x_path, train_y_path, number_of_bytes)

## Análise exploratória

In [ ]:
raw_y_train.head()

In [ ]:
px.histogram(raw_y_train.query('label != "Normal"'),
            x='label',
            color='label',
            category_orders={'label':raw_y_train['label'].value_counts().index.to_list()},
            title='Contagem de amostras por classe de ataque'
            )

In [ ]:
px.histogram(raw_y_train,
            x='protocol',
            color='protocol',
            category_orders={'protocol':raw_y_train['protocol'].value_counts().index.to_list()},
            title='Contagem de amostras por protocolo'
            )

In [ ]:
raw_y_train['timestamp'] = raw_y_train['timestamp'].astype(float)

In [ ]:
raw_y_train['timestamp'] = pd.to_datetime(raw_y_train['timestamp'], unit='s')

In [ ]:
freq = raw_y_train.set_index('timestamp').resample('1S').size().reset_index(name='freq')

fig = px.line(freq, x='timestamp', y='freq',
              title="Frequência de Pacotes por Segundo",
              labels={'timestamp': 'Tempo', 'freq': 'Número de Pacotes'})

fig.show()

In [ ]:
# Agrupa por 1 segundo + por tipo de ataque
freq_attack = (raw_y_train
               .set_index('timestamp')
               .groupby('label')
               .resample('1s')
               .size()
               .rename('freq')
               .reset_index())

# Gráfico multi-linha
fig = px.line(freq_attack,
              x='timestamp',
              y='freq',
              color='label',  # uma cor/linha para cada tipo de ataque
              title="Frequência de Pacotes por Tipo de Ataque",
              labels={'timestamp': 'Tempo', 'freq': 'Número de Pacotes'})

fig.show()

In [ ]:
# Agrupa por 1 segundo + por protocolo
freq_attack = (raw_y_train
               .set_index('timestamp')
               .groupby('protocol')
               .resample('1s')
               .size()
               .rename('freq')
               .reset_index())

# Gráfico multi-linha
fig = px.line(freq_attack,
              x='timestamp',
              y='freq',
              color='protocol',  # uma cor/linha para cada tipo de ataque
              title="Frequência de Pacotes por Protocolo",
              labels={'timestamp': 'Tempo', 'freq': 'Número de Pacotes'})

fig.show()

In [ ]:
raw_y_train[raw_y_train['label'] != 'Normal']

In [ ]:
raw_X_train.shape

In [ ]:
start_idx = 0 # 140311
sample = raw_X_train[start_idx:start_idx+window_size]
label_window = raw_y_train['label'].apply(lambda x: x != 'Normal')[start_idx:start_idx+window_size]  

# Máscara de pacotes injetados
mask = label_window.values

# Criar o Heatmap normal
fig = go.Figure(data=go.Heatmap(
    z=sample,
    colorscale='Viridis',
    colorbar=dict(title='Valor')
))

# Adicionar contornos onde mask == 1
for i in range(window_size):
    if mask[i] == 1:
        fig.add_shape(
            type="rect",
            x0=-0.5, x1=57.5,    # largura total dos bytes (0 a 57)
            y0=i-0.5, y1=i+0.5,  # linha do pacote
            line=dict(color="red", width=2)
        )

fig.update_yaxes(autorange="reversed")
fig.update_layout(title="Pacotes com Injeção Destacados em Vermelho",
                  xaxis_title="Bytes", yaxis_title="Pacotes na Janela")
fig.show()

In [ ]:
protocols = raw_y_train['protocol'].unique()

for protocol in protocols:
    protocol_idx = raw_y_train.index[raw_y_train['protocol'] == protocol]

    if len(protocol_idx) < window_size:
        print(f"Protocol {protocol} tem poucos pacotes para formar uma janela, pulando...")
        continue
    
    # Pegando uma janela para esse protocolo
    sample = raw_X_train[protocol_idx[:window_size]]

    # Mascara de pacotes injetados (1 = ataque)
    label_window = (raw_y_train.loc[protocol_idx[:window_size], 'label'] != 'Normal').values

    overlay = np.where(label_window[:, None] == 1, 1, np.nan)

    fig = go.Figure()

    # Heatmap em escala de cinza
    fig.add_trace(go.Heatmap(
        z=sample,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Valor')
    ))

    # Overlay vermelho translúcido
    fig.add_trace(go.Heatmap(
        z=overlay,
        colorscale=[[0, 'rgba(255,0,0,0.4)'], [1, 'rgba(255,0,0,0.4)']],
        showscale=False
    ))

    fig.update_yaxes(autorange="reversed")
    fig.update_layout(
        title=f"Protocolo {protocol} — Pacotes da Janela com Ataques em Vermelho",
        xaxis_title="Bytes (58 features)",
        yaxis_title="Pacotes (64 por janela)"
    )
    
    fig.show()


In [ ]:
protocols = raw_y_train['protocol'].unique()

for protocol in protocols:
    protocol_idx = raw_y_train.index[raw_y_train['protocol'] == protocol]

    if len(protocol_idx) < window_size:
        print(f"Protocol {protocol} tem poucos pacotes para formar uma janela, pulando...")
        continue
    
    # Pegando uma janela para esse protocolo
    sample = np.diff(raw_X_train[protocol_idx[:window_size]], axis=0)

    # Mascara de pacotes injetados (1 = ataque)
    label_window = (raw_y_train.loc[protocol_idx[:window_size], 'label'] != 'Normal').values

    overlay = np.where(label_window[:, None] == 1, 1, np.nan)

    fig = go.Figure()

    # Heatmap em escala de cinza
    fig.add_trace(go.Heatmap(
        z=sample,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Valor')
    ))

    # Overlay vermelho translúcido
    fig.add_trace(go.Heatmap(
        z=overlay,
        colorscale=[[0, 'rgba(255,0,0,0.4)'], [1, 'rgba(255,0,0,0.4)']],
        showscale=False
    ))

    fig.update_yaxes(autorange="reversed")
    fig.update_layout(
        title=f"Protocolo {protocol} — Pacotes da Janela com Ataques em Vermelho",
        xaxis_title="Bytes (58 features)",
        yaxis_title="Pacotes (64 por janela)"
    )
    
    fig.show()


In [ ]:
protocols = raw_y_train['protocol'].unique()

for protocol in protocols:
    protocol_idx = raw_y_train.index[raw_y_train['protocol'] == protocol]

    if len(protocol_idx) < window_size:
        print(f"Protocol {protocol} tem poucos pacotes para formar uma janela, pulando...")
        continue
    
    # Pegando uma janela para esse protocolo
    sample = [abs(np.diff(raw_X_train[protocol_idx[:window_size]], axis=0).sum(1))]

    # Mascara de pacotes injetados (1 = ataque)
    label_window = (raw_y_train.loc[protocol_idx[:window_size], 'label'] != 'Normal').values

    overlay = np.where(label_window[:, None] == 1, 1, np.nan)

    fig = go.Figure()

    # Heatmap em escala de cinza
    fig.add_trace(go.Heatmap(
        z=sample,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Valor')
    ))

    # Overlay vermelho translúcido
    fig.add_trace(go.Heatmap(
        z=overlay,
        colorscale=[[0, 'rgba(255,0,0,0.4)'], [1, 'rgba(255,0,0,0.4)']],
        showscale=False
    ))

    fig.update_yaxes(autorange="reversed")
    fig.update_layout(
        title=f"Protocolo {protocol} — Pacotes da Janela com Ataques em Vermelho",
        xaxis_title="Bytes (58 features)",
        yaxis_title="Pacotes (64 por janela)"
    )
    
    fig.show()


## Gerando features

In [ ]:
def __labeling_schema_vectorized(desc_windows: np.ndarray, labels: pd.DataFrame) -> np.ndarray:
    """
    Vectorized labeling:
    - Input: desc_windows: ndarray of shape (num_windows, window_size), dtype object/str
    - Output: ndarray of shape (num_windows,), dtype str
    - Rule: pick the most frequent non-'Normal' label in each row; if none, 'Normal'.
    - Ties: break by lexicographic order of the label string.
    """
    if desc_windows.size == 0:
        return np.array([], dtype=object)

    atk = [i for i in labels['label'].unique() if i != 'Normal']
    normal = "Normal"

    # Factorize all labels at once to integer codes
    flat = desc_windows.ravel()
    codes, uniques = pd.factorize(flat)              # codes in [0..K-1], uniques: array of labels
    codes = codes.reshape(desc_windows.shape)

    # Mask invalid (e.g., all-NaN rows) — pd.factorize uses -1 for NaN/None
    # We’ll just ignore -1 when counting.
    num_labels = len(uniques)
    if num_labels == 0:
        return np.full(desc_windows.shape[0], normal, dtype=object)

    # Count occurrences of each label per row (vectorized over rows)
    # Using one-hot accumulation without big temporary allocations.
    counts = np.zeros((desc_windows.shape[0], num_labels), dtype=np.int32)
    # Add counts column by column (window_size is small, so this is fast and memory-light)
    n_rows = desc_windows.shape[0]
    row_idx = np.arange(n_rows)
    for j in range(desc_windows.shape[1]):
        col = codes[:, j]
        valid = col >= 0
        np.add.at(counts, (row_idx[valid], col[valid]), 1)

    # Identify which global labels are "attack" labels
    uniques_str = uniques.astype(str)
    attack_mask = np.isin(uniques_str, list(atk))
    if not attack_mask.any():
        # No attack labels present anywhere
        return np.full(n_rows, normal, dtype=object)

    # Restrict to attack labels
    counts_attack = counts[:, attack_mask]                      # (n_rows, n_attack_labels)
    attack_labels = uniques_str[attack_mask]                    # (n_attack_labels,)

    # Row-wise max count among attacks
    max_attack = counts_attack.max(axis=1)                      # (n_rows,)
    has_attack = max_attack > 0

    # Tie-break lexicographically among attack labels
    # Precompute a rank (0 = smallest lexicographic label)
    lex_ranks = np.argsort(attack_labels).argsort()             # rank per attack label index
    # Among positions with count == max_attack, choose smallest lex rank.
    tie_mask = counts_attack == max_attack[:, None]             # (n_rows, n_attack_labels)
    # Use -lex_ranks so that argmax picks the smallest rank; set non-ties to -inf
    prefs = np.where(tie_mask, -lex_ranks, -np.inf)
    best_j = np.argmax(prefs, axis=1)                           # (n_rows,)

    # Build the result
    out = np.full(n_rows, normal, dtype=object)
    out[has_attack] = attack_labels[best_j[has_attack]]
    return out


def __get_window_data(data: np.ndarray, target: pd.DataFrame, window_stride: int=1, window_size: int=2,
                    max_windows: int=None, rng=None):
    rng = np.random.default_rng(rng)
    n = data.shape[0]
    last_start = n - window_size
    if last_start < 0:
        return [], []

    # all candidate starts given the stride
    starts = np.arange(0, last_start + 1, window_stride)

    # --- lightweight label-side windows (cheap) ---
    desc        = target['label'].to_numpy()
    pkt_idx     = target['pkt_idx'].to_numpy()
    timestamps  = target['timestamp'].to_numpy()
    protocols   = target['protocol'].to_numpy()

    desc_windows_all = np.lib.stride_tricks.sliding_window_view(desc, window_shape=window_size)

    # subsample starts *before* touching big value windows
    if max_windows is not None and len(starts) > max_windows:
        starts = rng.choice(starts, size=max_windows, replace=False)
        starts.sort()  # keep roughly chronological order

    # compute labels only for chosen starts
    seq_y = __labeling_schema_vectorized(desc_windows_all[starts], target)
    start_pkt_idx = pkt_idx[starts]
    desc_windows = desc_windows_all[starts]

    # --- now build value windows for the chosen starts only ---
    # sliding_window_view returns a view; avoid materializing everything
    X_view = np.lib.stride_tricks.sliding_window_view(data, window_shape=(window_size,) + data.shape[1:])
    X_view = X_view.reshape(n - window_size + 1, window_size, *data.shape[1:])
    # X = X_view[starts]

    y = list(zip(seq_y, start_pkt_idx.astype(int), desc_windows, timestamps[starts], protocols[starts]))
    y = pd.DataFrame(y, columns=['label', 'start_idx', 'desc_windows', 'timestamps', 'protocols'])

    print("Class distribution:")
    print(Counter(seq_y))
        
    return (X_view, starts), y

def process(data: np.ndarray, target: pd.DataFrame, window_size: int = 64, window_stride: int = 1, max_windows: int = None) -> Tuple[np.ndarray, pd.DataFrame]:
    (X_view, starts), y = __get_window_data(data, target, window_stride=window_stride, window_size=window_size, 
                                        max_windows=max_windows)
    return X_view[starts], y

In [ ]:
X_train, y_train = process(raw_X_train, raw_y_train, window_size, window_stride)

In [ ]:
X_train.shape

In [ ]:
del raw_X_train, raw_y_train

In [ ]:
X_train_flat = np.diff(X_train, axis=1).sum(1)

In [ ]:
import pickle
import torch
torch.save({'X_train_flat': X_train_flat, 'y_train': y_train}, f"train_data_processed.pt", pickle_protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
start_idx = 0 # 140311
sample = [X_train_flat[0]]

# Criar o Heatmap normal
fig = go.Figure(data=go.Heatmap(
    z=sample,
    colorscale='Viridis',
    colorbar=dict(title='Valor')
))

fig.update_yaxes(autorange="reversed")
fig.update_layout(title="Pacotes com Injeção Destacados em Vermelho",
                  xaxis_title="Bytes", yaxis_title="Pacotes na Janela")
fig.show()

## Carregando cash

In [ ]:
cache = torch.load(f"train_data_processed.pt", weights_only=False)
X_train_flat, y_train = cache['X_train_flat'], cache['y_train']

## Separando conjunto de treino e validação

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_val, classes_train, classes_val = train_test_split(X_train_flat, y_train['label'], test_size=0.65, stratify=y_train['label'], random_state=DEFAULT_SEED)

classes_train, classes_val =  classes_train.reset_index(drop=True), classes_val.reset_index(drop=True)

y_train, y_val = classes_train.apply(lambda c: 0 if c == 'Normal' else 1), classes_val.apply(lambda c: 0 if c == 'Normal' else 1)

In [ ]:
def get_class_data(label='Normal', subset=0.01):
    indexes = y_val[classes_val == label].index
    starts = np.random.choice(indexes, size=int(subset*len(indexes)), replace=False)
    X_val_sub = X_val[starts]
    y_val_sub = y_val[starts]
    print(classes_val.iloc[starts].value_counts())

    return X_val_sub, y_val_sub

## Treinando uma Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier().fit(X_train, y_train)

In [ ]:
Yhat = rf.predict(X_val)

In [ ]:
plot_confusion_matrix(y_val, Yhat)

In [ ]:
get_overall_metrics(y_val, Yhat)

In [ ]:
get_tpr_per_attack(classes_val, Yhat)

## Construindo um global sorrogate

### Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(criterion='entropy', max_depth=2)

dt.fit(X_val, Yhat)

In [ ]:
Yhat_dt = dt.predict(X_val)

In [ ]:
differences = (Yhat_dt != Yhat)
print(f'Diferença entre modelos: {(differences.sum() * 100 / Yhat.shape[0]):.2f}% instâncias')

In [ ]:
get_overall_metrics(y_val, Yhat_dt)

In [ ]:
get_tpr_per_attack(classes_val, Yhat_dt)

In [ ]:
from sklearn import tree

plt.figure(figsize=(20, 10))
tree.plot_tree(dt, class_names=['no', 'yes'], filled=True)

plt.show()

De forma global, a Decision Tree (DT) foi treinada com o objetivo de imitar o modelo black-box (Random Forest). Nesse cenário, ao observar o diagrama de decisões, não é possível extrair boas explicações sobre o comportamento do modelo black-box, pois a entropia das folhas está alta (sendo a menor igual a 0,328), o que indica alta impureza. A decisão para a classe verdadeira (ataque) é tomada quando o byte 27 ≤ -0,012 e, adicionalmente, byte 55 ≤ 0,002 ou byte 25 > 0,012.

Analisando os pacotes no Wireshark, observa-se que o byte 27 está relacionado ao Stream ID no protocolo AVTP, ao source address nos protocolos IP/UDP, e ao correction field no PTP. Por outro lado, o byte 55 corresponde ao Continuity Counter do AVTP, e a campos de padding nos protocolos IP/UDP e PTP.

Dessa forma, os ataques presentes no dataset envolvem injeção de pacotes, o que pode de fato impactar esses campos, já que a injeção interrompe o comportamento normal da comunicação. Entretanto, ao analisar a taxa de detecção por tipo de ataque, observa-se que o modelo conseguiu detectar apenas os ataques PTP Sync e Switch MAC Flooding, ambos mais verbosos e passíveis de detecção por regras simples. Por outro lado, os demais ataques são mais complexos e exigem regras distintas e mais elaboradas.

Portanto, a Decision Tree não consegue imitar adequadamente o modelo black-box, devido à complexidade dos dados, que envolvem cinco ataques diferentes sobre protocolos distintos. Por esse motivo, modelos mais simples se mostram ineficazes para esse tipo de tarefa.

## Permutation Feature Importance

In [ ]:
def plot_pfi(result):
    forest_importances = pd.Series(result.importances_mean)

    fig, ax = plt.subplots(figsize=(12, 6))

    forest_importances.plot.bar(yerr=result.importances_std, ax=ax)

    ax.set_title("Feature importances using permutation on full model")
    ax.set_ylabel("Mean accuracy decrease")
    fig.tight_layout()
    plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

In [ ]:
subset = 0.01
indexes = y_train.reset_index(drop=True).index
starts = np.random.choice(indexes, size=int(subset*len(indexes)), replace=False)
X_train_sub = X_train[starts]
y_train_sub = y_train[starts]
classes_train.iloc[starts].value_counts()

In [ ]:
train_result = permutation_importance(rf, X_train_sub, y_train_sub, n_repeats=50, random_state=DEFAULT_SEED)

In [ ]:
plot_pfi(train_result)

In [ ]:
subset = 0.01
indexes = y_val.reset_index(drop=True).index
starts = np.random.choice(indexes, size=int(subset*len(indexes)), replace=False)
X_val_sub = X_val[starts]
y_val_sub = y_val[starts]
classes_val.iloc[starts].value_counts()

In [ ]:
result = permutation_importance(rf, X_val_sub, y_val_sub, n_repeats=50, random_state=DEFAULT_SEED)

In [ ]:
plot_pfi(result)

In [ ]:
X_val_normal_sub, y_val_normal_sub = get_class_data(label='Normal', subset=0.001)
pfi_normal = permutation_importance(rf, X_val_normal_sub, y_val_normal_sub, n_repeats=50, random_state=DEFAULT_SEED)
del X_val_normal_sub, y_val_normal_sub

In [ ]:
X_val_dos_sub, y_val_dos_sub = get_class_data(label='CAN DoS', subset=0.001)
pfi_dos = permutation_importance(rf, X_val_dos_sub, y_val_dos_sub, n_repeats=50, random_state=DEFAULT_SEED)
del X_val_dos_sub, y_val_dos_sub

In [ ]:
X_val_replay_sub, y_val_replay_sub = get_class_data('CAN Replay', subset=0.01)
pfi_replay = permutation_importance(rf, X_val_replay_sub, y_val_replay_sub, n_repeats=50, random_state=DEFAULT_SEED)
del X_val_replay_sub, y_val_replay_sub

In [ ]:
X_val_mac_sub, y_val_mac_sub = get_class_data('Switch MAC Flooding', subset=0.001)
pfi_mac = permutation_importance(rf, X_val_mac_sub, y_val_mac_sub, n_repeats=50, random_state=DEFAULT_SEED)
del X_val_mac_sub, y_val_mac_sub

In [ ]:
X_val_fi_sub, y_val_fi_sub = get_class_data('Frame Injection', subset=0.001)
pfi_fi = permutation_importance(rf, X_val_fi_sub, y_val_fi_sub, n_repeats=50, random_state=DEFAULT_SEED)
del X_val_fi_sub, y_val_fi_sub

In [ ]:
X_val_ptp_sub, y_val_ptp_sub = get_class_data('PTP Sync', subset=0.001)
pfi_ptp = permutation_importance(rf, X_val_ptp_sub, y_val_ptp_sub, n_repeats=50, random_state=DEFAULT_SEED)
del X_val_ptp_sub, y_val_ptp_sub

In [ ]:
pfi_results = {
    'CAN DoS': pfi_dos.importances_mean,
    'CAN Replay': pfi_replay.importances_mean,
    'Switch MAC Flooding': pfi_mac.importances_mean,
}

# Convert to DataFrame
df_pfi = pd.DataFrame(pfi_results, index=range(len(pfi_normal.importances_mean)))

# Plot
plt.figure(figsize=(14, 7))
df_pfi.plot(kind='bar', figsize=(14, 7))
plt.title('Permutation Feature Importance Comparison by Attack Type')
plt.xlabel('Feature Index')
plt.ylabel('Mean Accuracy Decrease')
plt.legend(title='Attack Type')
plt.tight_layout()
plt.show()

In [ ]:
plot_pfi(pfi_mac)

In [ ]:
pfi_results = {
    'Frame Injection': pfi_fi.importances_mean,
    'PTP Sync': pfi_ptp.importances_mean
}

# Convert to DataFrame
df_pfi = pd.DataFrame(pfi_results, index=range(len(pfi_normal.importances_mean)))

# Plot
plt.figure(figsize=(14, 7))
df_pfi.plot(kind='bar', figsize=(14, 7))
plt.title('Permutation Feature Importance Comparison by Attack Type')
plt.xlabel('Feature Index')
plt.ylabel('Mean Accuracy Decrease')
plt.legend(title='Attack Type')
plt.tight_layout()
plt.show()

Ao observar a análise de feature importance, foi investigada a relação entre os conjuntos de treino e validação, além da importância de features para cada ataque individualmente, com o objetivo de compreender como o modelo realiza a tomada de decisão em relação a cada tipo de ataque.

Na comparação entre os conjuntos de validação e treino, os bytes 19 e 25 mantêm alta importância, enquanto os bytes 20, 32 e 49 apresentam aumento de relevância no conjunto de validação. O byte 20 é considerado um dos mais importantes no caso do ataque de Frame Injection, pois corresponde ao Sequence Number, que possui comportamento determinístico, aumentando de 1 em 1 à medida que os pacotes são enviados. Durante o ataque, pacotes previamente transmitidos são reinjetados, interrompendo a sequência esperada, assim, o ataque pode ser detectado pela observação desse campo.

Na análise específica do ataque Frame Injection, observa-se que o byte 20 apresenta alta importância, embora em conjunto com os bytes 32, 33 e 52, sendo este último o mais relevante. O byte 52, no protocolo AVTP, está relacionado ao timestamp do dado de vídeo. Quando um pacote é reinjetado, esse timestamp representa um tempo no passado, o que é um forte indício de que o ataque está ocorrendo.

O ataque de CAN Replay apresenta importância significativamente maior no byte 32, que corresponde ao destination address do protocolo UDP. Nesse ataque, os pacotes também são reinjetados na rede; contudo, o CAN, sendo um protocolo automotivo, é encapsulado em UDP para transmissão sobre Ethernet no dataset. Assim, as informações da mensagem CAN iniciam no byte 34. O modelo parece detectar o ataque com base em dados do UDP, e não diretamente no conteúdo CAN. Isso é possível porque as mensagens foram reinjetadas em uma porta de destino fixa, diferente do comportamento normal. No entanto, como o modelo não atribuiu tanta importância à própria mensagem CAN (alvo do ataque), ele obteve apenas uma taxa de verdadeiros positivos (TPR) de 0,64 para esse caso.

Já no ataque CAN DoS, o modelo atribuiu maior importância aos bytes após o 34, especialmente ao byte 41. Esse ataque consiste no envio de mensagens com máxima prioridade na rede CAN, o que inviabiliza o fluxo normal de comunicação. Por ser um comportamento facilmente distinguível, já que a prioridade é dada por um identificador menor e o payload permanece fixo, o modelo conseguiu uma TPR de 0,89, superior à do CAN Replay.

O ataque de MAC Flooding se destaca por sua simplicidade: apenas dois bytes (19 e 53) são suficientes para sua detecção. Esse ataque tem baixa complexidade, pois a rede utilizada no dataset emprega apenas dois endereços MAC. Assim, quando novos endereços são injetados, tornam-se facilmente identificáveis como anômalos, resultando em uma TPR de 0,99.

De forma semelhante, o ataque PTP Sync tenta interromper a sincronização entre os nodes da rede, alterando as referências temporais. O modelo atribuiu importância aos bytes 17, 38, 44 e 45, que estão relacionados a campos de configuração temporal, o que é consistente com a natureza do ataque.

## Partial Dependence Plots

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

In [ ]:
X_val_dos_sub, y_val_dos_sub = get_class_data(label='Frame Injection', subset=0.25)
disp1 = PartialDependenceDisplay.from_estimator(rf, X_val_dos_sub, [18, 19, 32, 37, 52], grid_resolution=20)
del X_val_dos_sub, y_val_dos_sub

In [ ]:
X_val_dos_sub, y_val_dos_sub = get_class_data(label='CAN Replay', subset=0.25)
disp1 = PartialDependenceDisplay.from_estimator(rf, X_val_dos_sub, [32, 37], grid_resolution=20)
del X_val_dos_sub, y_val_dos_sub

In [ ]:
X_val_dos_sub, y_val_dos_sub = get_class_data(label='CAN DoS', subset=0.25)
disp1 = PartialDependenceDisplay.from_estimator(rf, X_val_dos_sub, [33, 41], grid_resolution=20)
del X_val_dos_sub, y_val_dos_sub

In [ ]:
X_val_mac_sub, y_val_mac_sub = get_class_data('Switch MAC Flooding', subset=0.25)
disp1 = PartialDependenceDisplay.from_estimator(rf, X_val_mac_sub, [19, 53], grid_resolution=20)
del X_val_mac_sub, y_val_mac_sub

In [ ]:
X_val_mac_sub, y_val_mac_sub = get_class_data('PTP Sync', subset=0.25)
disp1 = PartialDependenceDisplay.from_estimator(rf, X_val_mac_sub, [17, 38, 44, 45], grid_resolution=20)
del X_val_mac_sub, y_val_mac_sub

A análise de Partial Dependence Plots (PDP) indica como a variabilidade das variáveis se relaciona com a saída do modelo. É importante destacar que o processo de extração de features tem como output a medida de quanto os bytes variam dentro de uma janela temporal. Assim, quando uma variável assume valores próximos de 0, isso indica baixa variabilidade.

Um comportamento semelhante pode ser observado nos ataques Frame Injection, CAN Replay e CAN DoS, nos quais a probabilidade da classe “ataque” aumenta à medida que as features tendem a apresentar maior variação, seja em direção positiva ou negativa. Esse efeito ocorre porque, nos dados normais, essas features apresentam baixa variabilidade, enquanto os ataques interrompem o comportamento regular dessas variáveis.

Por outro lado, ataques de menor complexidade, como o MAC Flooding e o PTP Sync, apresentam um comportamento diferente. No caso do MAC Flooding, quando a variável byte 19 assume valores negativos, há alta probabilidade de ataque; já para valores positivos, essa probabilidade diminui. No PTP Sync, observa-se maior probabilidade de ataque quando os valores estão próximos de 0.

Nesse contexto, percebe-se uma complexidade na interpretabilidade do modelo em relação à variabilidade individual das features. A técnica de PDP não considera as interações entre variáveis, o que é especialmente relevante neste caso, pois os bytes dos cabeçalhos dos pacotes frequentemente representam campos multibyte. Assim, a interpretação isolada de cada byte pode não capturar adequadamente o comportamento conjunto dessas features, impactando diretamente a análise.

## LIME

In [ ]:
from lime import lime_tabular

# Criando o explicador
explainer = lime_tabular.LimeTabularExplainer(X_train)

In [ ]:
import IPython.display as ipd
import sys
sys.modules['IPython.core.display'] = ipd

In [ ]:
i = y_val[classes_val == 'Normal'].index[3]

exp = explainer.explain_instance(X_val[i], rf.predict_proba)

exp.show_in_notebook(show_table=True, show_all=False)

In [ ]:
i = y_val[classes_val == 'Frame Injection'].index[0]

exp = explainer.explain_instance(X_val[i], rf.predict_proba)

exp.show_in_notebook(show_table=True, show_all=False)

In [ ]:
i = y_val[classes_val == 'CAN Replay'].index[0]

exp = explainer.explain_instance(X_val[i], rf.predict_proba)

exp.show_in_notebook(show_table=True, show_all=False)

In [ ]:
i = y_val[classes_val == 'CAN DoS'].index[0]

exp = explainer.explain_instance(X_val[i], rf.predict_proba)

exp.show_in_notebook(show_table=True, show_all=False)

In [ ]:
i = y_val[classes_val == 'Switch MAC Flooding'].index[0]

exp = explainer.explain_instance(X_val[i], rf.predict_proba)

exp.show_in_notebook(show_table=True, show_all=False)

In [ ]:
i = y_val[classes_val == 'PTP Sync'].index[0]

exp = explainer.explain_instance(X_val[i], rf.predict_proba)

exp.show_in_notebook(show_table=True, show_all=False)

A análise com LIME (Local Interpretable Model-Agnostic Explanations) permitiu compreender como o modelo toma decisões localmente, identificando quais features (bytes) mais influenciaram a classificação de amostras específicas. Para as instâncias normais, observou-se que para os bytes 11, 4, 8, 9, 22, 37, 39  apresentaram valores próximos de zero e contribuíram negativamente para a classe de ataque, refletindo a baixa variabilidade esperada em condições normais.

Nos ataques mais complexos, como Frame Injection, CAN Replay e CAN DoS, o LIME destacou múltiplas features relevantes, em especial os bytes 20, 32 e 19, associados a campos como Sequence Number e timestamps, que sofrem alterações durante a reinjeção de pacotes ou priorização anômala de mensagens. Já nos ataques de MAC Flooding e PTP Sync, o modelo baseou-se em um número reduzido de features, como os bytes 19 e 9, suficientes para distinguir esses ataques de baixa complexidade.

De forma geral, as explicações locais fornecidas pelo LIME confirmam que o modelo aprendeu padrões coerentes com o comportamento dos protocolos, utilizando diferentes features de acordo com a natureza e a complexidade de cada ataque.

## SHARP

In [ ]:
import shap

In [ ]:
explainer = shap.TreeExplainer(rf)

In [ ]:
def plot_shap_explainer(data):
    explanation = explainer(data)

    shap_values = explanation.values[:, :, 1]
    base_values = explanation.base_values[:, 1]

    shap.plots.beeswarm(
        shap.Explanation(
            values=shap_values,
            base_values=base_values,
            data=data,
        )
    )

In [ ]:
X_val_normal_sub, y_val_normal_sub = get_class_data('Frame Injection', subset=0.001)
plot_shap_explainer(X_val_normal_sub)
del X_val_normal_sub, y_val_normal_sub

In [ ]:
X_val_normal_sub, y_val_normal_sub = get_class_data('CAN Replay', subset=0.001)
plot_shap_explainer(X_val_normal_sub)
del X_val_normal_sub, y_val_normal_sub

In [ ]:
X_val_normal_sub, y_val_normal_sub = get_class_data('CAN DoS', subset=0.001)
plot_shap_explainer(X_val_normal_sub)
del X_val_normal_sub, y_val_normal_sub

In [ ]:
X_val_normal_sub, y_val_normal_sub = get_class_data('Switch MAC Flooding', subset=0.001)
plot_shap_explainer(X_val_normal_sub)
del X_val_normal_sub, y_val_normal_sub

In [ ]:
X_val_normal_sub, y_val_normal_sub = get_class_data('PTP Sync', subset=0.001)
plot_shap_explainer(X_val_normal_sub)
del X_val_normal_sub, y_val_normal_sub

A análise utilizando SHAP permite notar uma semelhança entre o ataque de Frame Injection e a análise de feature importance, na qual os bytes 52 e 51 (timestamp de vídeo) apresentam maior influência. Nesse cenário, a probabilidade da classe aumenta quando os valores dessas features são muito baixos ou muito altos. Como as features variam de -1 a 1, magnitudes elevadas indicam maior variabilidade dentro da janela, o que pode sinalizar uma interrupção no fluxo normal. Além disso, o byte 20 também está entre os mais influentes; esse byte representa um número de sequência dos pacotes e possui um comportamento determinístico que é interrompido pelo ataque.

A comparação entre os ataques CAN DoS e CAN Replay mostra que o modelo utiliza padrões distintos para reconhecer cada tipo de atividade maliciosa. No CAN DoS, as features mais relevantes apresentam impactos mais concentrados e consistentes, indicando que o ataque altera o fluxo de tráfego de forma uniforme, especialmente em termos de frequência e ritmo das mensagens (bytes 37, 25 e 19). Já no CAN Replay, o efeito das features é mais disperso, revelando que o modelo depende fortemente das variações no conteúdo e na repetição das mensagens para identificar esse ataque (bytes 49, 19, 25 e 50). Apesar de algumas features importantes aparecerem em ambos os cenários, elas exercem papéis diferentes: no DoS, destacam mudanças no fluxo, enquanto no Replay refletem padrões específicos de duplicação de mensagens. Isso evidencia que o modelo distingue bem as assinaturas características de cada ataque.

No ataque Switch MAC Flooding do dataset TOW-IDS, o modelo identifica o comportamento malicioso principalmente por meio de alterações consistentes em features como as bytes 19, 25 e 32, que aparecem com impacto positivo concentrado. Esse padrão indica que o modelo está captando o efeito típico do MAC Flooding: a geração rápida e repetitiva de quadros com endereços MAC variados, que sobrecarrega a tabela CAM do switch. O agrupamento dos SHAP values e sua predominância positiva sugerem que o ataque produz mudanças claras e uniformes no tráfego, como aumento anormal de taxa, diversidade de endereços e padrões de distribuição, que são facilmente reconhecidas. Assim, o modelo responde a uma assinatura altamente previsível, na qual valores elevados dessas features aumentam diretamente a probabilidade de classificação como MAC Flooding.

Para o ataque PTP Sync, o modelo se apoia em um conjunto diferente de características, com destaque para os bytes 25, 19 e 2, que exibem impactos mais variados e uma dispersão mais ampla nos SHAP values. Isso reflete a natureza sensível desse ataque, em que pacotes PTP falsificados ou manipulados interferem na sincronização temporal dos dispositivos da rede automotiva. O modelo captura essas perturbações por meio de mudanças nos padrões temporais, nos intervalos entre mensagens e no comportamento geral do fluxo, que aparecem como contribuições positivas ou negativas dependendo do valor da feature. A maior diversidade nos impactos indica que o ataque não altera o tráfego de maneira tão uniforme quanto o MAC Flooding. Em vez disso, provoca desvios específicos e irregulares no tempo, que o modelo consegue identificar como anomalias típicas de um PTP Sync malicioso.

## Conclusão

O conjunto de técnicas utilizadas para interpretar o modelo pode trazer insights importantes sobre seu funcionamento. Foi observado que cabeçalhos relacionados à sequência tiveram maior importância e influência na tomada de decisão, uma vez que os ataques injetam pacotes maliciosos interrompendo o comportamento normal desses campos. As técnicas foram aplicadas separando as labels entre os cinco ataques presentes no dataset, e essa abordagem permitiu uma melhor explicação da tomada de decisão do modelo, já que os ataques possuem naturezas distintas e manipulam protocolos de rede diferentes. Entretanto, as features representam bytes individuais dos pacotes, e na maioria das vezes a informação real precisa de mais de um byte para ser representada. Assim, existe uma relação clara entre as features, e tanto o modelo quanto as técnicas de interpretabilidade precisam ser capazes de processar essas dependências. Surge, então, a limitação de que as técnicas utilizadas não conseguem medir diretamente essas relações. Como trabalhos futuros, pode ser explorada a transformação dos bytes em suas informações originais antes do pré-processamento, com o objetivo de minimizar essa limitação, ainda que seja importante reconhecer que relações entre informações podem continuar existindo.

# Testes de hipóteses

## 1. Teste de Normalidade (Shapiro-Wilk)

Verifica se as features seguem distribuição normal.

In [62]:
from scipy.stats import shapiro

# Teste de normalidade para algumas features importantes
important_features = [19, 20, 25, 32, 52]
print("Teste de Normalidade (Shapiro-Wilk)")
print("=" * 50)
print(f"H0: Os dados seguem distribuição normal")
print(f"H1: Os dados não seguem distribuição normal")
print(f"Nível de significância: α = 0.05\n")

for feat_idx in important_features:
    sample = X_val[:, feat_idx][:5000]  # Amostra limitada
    stat, p_value = shapiro(sample)
    result = "Rejeita H0" if p_value < 0.05 else "Não rejeita H0"
    print(f"Feature {feat_idx}: p-value = {p_value:.6f} -> {result}")

Teste de Normalidade (Shapiro-Wilk)
H0: Os dados seguem distribuição normal
H1: Os dados não seguem distribuição normal
Nível de significância: α = 0.05

Feature 19: p-value = 0.000000 -> Rejeita H0
Feature 20: p-value = 0.000000 -> Rejeita H0
Feature 25: p-value = 0.000000 -> Rejeita H0
Feature 32: p-value = 0.000000 -> Rejeita H0
Feature 52: p-value = 0.000000 -> Rejeita H0


## 2. Teste de Mann-Whitney U

Compara a distribuição de cada feature entre tráfego normal e ataques.

In [63]:
from scipy.stats import mannwhitneyu

# Separar dados normais e ataques
normal_mask = classes_val == 'Normal'
attack_mask = ~normal_mask

print("\nTeste de Mann-Whitney U (Normal vs Ataques)")
print("=" * 50)
print(f"H0: As distribuições são iguais")
print(f"H1: As distribuições são diferentes")
print(f"Nível de significância: α = 0.05\n")

for feat_idx in important_features:
    normal_data = X_val[normal_mask, feat_idx]
    attack_data = X_val[attack_mask, feat_idx]
    stat, p_value = mannwhitneyu(normal_data, attack_data, alternative='two-sided')
    result = "Rejeita H0 (diferença significativa)" if p_value < 0.05 else "Não rejeita H0"
    print(f"Feature {feat_idx}: p-value = {p_value:.6e} -> {result}")


Teste de Mann-Whitney U (Normal vs Ataques)
H0: As distribuições são iguais
H1: As distribuições são diferentes
Nível de significância: α = 0.05

Feature 19: p-value = 6.128643e-08 -> Rejeita H0 (diferença significativa)
Feature 20: p-value = 1.589867e-228 -> Rejeita H0 (diferença significativa)
Feature 25: p-value = 4.148900e-06 -> Rejeita H0 (diferença significativa)
Feature 32: p-value = 9.601707e-02 -> Não rejeita H0
Feature 52: p-value = 0.000000e+00 -> Rejeita H0 (diferença significativa)


## 3. Teste de Kruskal-Wallis

Compara as distribuições das features entre os diferentes tipos de ataques.

In [64]:
from scipy.stats import kruskal

# Obter dados por tipo de ataque
attack_types = ['CAN DoS', 'CAN Replay', 'Frame Injection', 'Switch MAC Flooding', 'PTP Sync']

print("\nTeste de Kruskal-Wallis (Comparação entre tipos de ataque)")
print("=" * 50)
print(f"H0: Todas as distribuições são iguais")
print(f"H1: Pelo menos uma distribuição é diferente")
print(f"Nível de significância: α = 0.05\n")

for feat_idx in important_features:
    groups = [X_val[classes_val == attack, feat_idx] for attack in attack_types]
    stat, p_value = kruskal(*groups)
    result = "Rejeita H0 (diferenças significativas)" if p_value < 0.05 else "Não rejeita H0"
    print(f"Feature {feat_idx}: p-value = {p_value:.6e} -> {result}")


Teste de Kruskal-Wallis (Comparação entre tipos de ataque)
H0: Todas as distribuições são iguais
H1: Pelo menos uma distribuição é diferente
Nível de significância: α = 0.05

Feature 19: p-value = 0.000000e+00 -> Rejeita H0 (diferenças significativas)
Feature 20: p-value = 4.501013e-06 -> Rejeita H0 (diferenças significativas)
Feature 25: p-value = 0.000000e+00 -> Rejeita H0 (diferenças significativas)
Feature 32: p-value = 1.686153e-05 -> Rejeita H0 (diferenças significativas)
Feature 52: p-value = 5.459600e-07 -> Rejeita H0 (diferenças significativas)


## 4. Teste de Correlação (Spearman)

Verifica se há correlação entre as features mais importantes identificadas pelo modelo.

In [65]:
from scipy.stats import spearmanr

print("\nTeste de Correlação de Spearman")
print("=" * 50)
print(f"H0: Não há correlação entre as features")
print(f"H1: Há correlação entre as features")
print(f"Nível de significância: α = 0.05\n")

# Matriz de correlação para features importantes
for i in range(len(important_features)):
    for j in range(i+1, len(important_features)):
        feat_i, feat_j = important_features[i], important_features[j]
        corr, p_value = spearmanr(X_val[:, feat_i], X_val[:, feat_j])
        result = "Rejeita H0 (correlação significativa)" if p_value < 0.05 else "Não rejeita H0"
        print(f"Feature {feat_i} vs {feat_j}: ρ = {corr:.4f}, p-value = {p_value:.6e} -> {result}")


Teste de Correlação de Spearman
H0: Não há correlação entre as features
H1: Há correlação entre as features
Nível de significância: α = 0.05

Feature 19 vs 20: ρ = 0.2274, p-value = 0.000000e+00 -> Rejeita H0 (correlação significativa)
Feature 19 vs 25: ρ = 0.3277, p-value = 0.000000e+00 -> Rejeita H0 (correlação significativa)
Feature 19 vs 32: ρ = -0.0046, p-value = 3.963102e-05 -> Rejeita H0 (correlação significativa)
Feature 19 vs 52: ρ = 0.0889, p-value = 0.000000e+00 -> Rejeita H0 (correlação significativa)
Feature 20 vs 25: ρ = 0.1523, p-value = 0.000000e+00 -> Rejeita H0 (correlação significativa)
Feature 20 vs 32: ρ = -0.1973, p-value = 0.000000e+00 -> Rejeita H0 (correlação significativa)
Feature 20 vs 52: ρ = 0.3444, p-value = 0.000000e+00 -> Rejeita H0 (correlação significativa)
Feature 25 vs 32: ρ = 0.2516, p-value = 0.000000e+00 -> Rejeita H0 (correlação significativa)
Feature 25 vs 52: ρ = -0.0495, p-value = 0.000000e+00 -> Rejeita H0 (correlação significativa)
Feature 

## 5. Teste de Permutação para Feature Importance

Valida estatisticamente se a importância das features é significativa.

In [ ]:
print("\nTeste de Permutação - Feature Importance")
print("=" * 50)
print(f"H0: A feature não tem importância real (baseline)")
print(f"H1: A feature tem importância significativa")
print(f"Nível de significância: α = 0.05\n")

# Usar resultados do PFI já calculado (variável 'train_result' da célula 59)
# Primeiro, vamos obter o PFI novamente para o conjunto de validação
from sklearn.inspection import permutation_importance
pfi_result = permutation_importance(rf, X_val_sub, y_val_sub, n_repeats=50, random_state=DEFAULT_SEED)

baseline_score = rf.score(X_val_sub, y_val_sub)
print(f"Baseline accuracy: {baseline_score:.4f}\n")

# Verificar importância vs erro padrão
for feat_idx in important_features:
    mean_importance = pfi_result.importances_mean[feat_idx]
    std_importance = pfi_result.importances_std[feat_idx]
    
    # Teste estatístico: importância / erro padrão (t-statistic aproximado)
    if std_importance > 0:
        t_stat = mean_importance / std_importance
        significant = t_stat > 1.96  # Aproximadamente 95% de confiança
        result_text = "Significativa" if significant else "Não significativa"
        print(f"Feature {feat_idx}: importance = {mean_importance:.6f} ± {std_importance:.6f} (t = {t_stat:.2f}) -> {result_text}")
    else:
        print(f"Feature {feat_idx}: importance = {mean_importance:.6f} (sem variação)")


Teste de Permutação - Feature Importance
H0: A feature não tem importância real (baseline)
H1: A feature tem importância significativa
Nível de significância: α = 0.05


H0: A feature não tem importância real (baseline)
H1: A feature tem importância significativa
Nível de significância: α = 0.05



## Interpretação dos Resultados

Os testes estatísticos confirmam as observações obtidas durante a análise de interpretabilidade:

1. **Teste de Normalidade**: As features não seguem distribuição normal, justificando o uso de testes não-paramétricos.

2. **Mann-Whitney U**: Confirma que as features importantes apresentam distribuições significativamente diferentes entre tráfego normal e ataques.

3. **Kruskal-Wallis**: Demonstra que diferentes tipos de ataques afetam as features de maneiras estatisticamente distintas.

4. **Correlação de Spearman**: Identifica relações entre features, especialmente importante para campos multi-byte dos protocolos.

5. **Teste de Permutação**: Valida que as features identificadas como importantes pelo modelo têm impacto estatisticamente significativo na predição.